# OceanScan - Model Training (YOLOv8n)

Trains a **YOLOv8n** nano object detector on the **noise-filtered** sonar images in `output/noise_filter/train_noise_filter/`.

- **Train set**: `output/noise_filter/train_noise_filter/train` (402 images)
- **Val set**:   `output/noise_filter/train_noise_filter/valid` (110 images)
- **Classes**:   `0 body`, `1 plane`, `2 ship`, `3 target` (YOLO format)
- **Early stopping**: `patience=15` -> stops automatically when val mAP plateaus
- **Device**: CPU-friendly (imgsz=640, batch=8)

At the end, the best weights are saved to `backend/best.pt` for the prediction notebook.

## 0. Dependencies

"Run All" auto-installs anything that is missing (e.g. `cv2`), so no manual `pip install` is needed.

In [ ]:
import importlib
import subprocess
import sys

print("Kernel Python:", sys.executable)

_DEP_MAP = {
    "numpy": "numpy",
    "cv2": "opencv-python",
    "matplotlib": "matplotlib",
    "pandas": "pandas",
    "tqdm": "tqdm",
    "ultralytics": "ultralytics",
}

missing = []
for _mod, _pkg in _DEP_MAP.items():
    try:
        importlib.import_module(_mod)
    except ImportError:
        missing.append(_pkg)

if missing:
    print("Installing missing packages:", ", ".join(sorted(missing)))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *sorted(missing)])

failed = []
for _mod in _DEP_MAP:
    try:
        importlib.import_module(_mod)
    except ImportError:
        failed.append(_mod)

if failed:
    print("!!! STILL MISSING:", failed)
    print("Open a terminal here and run:  pip install -r requirements.txt")
else:
    print("All dependencies OK.")

## 1. Setup

In [ ]:
import shutil
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO

ROOT = Path(r"D:\1. Project Program\1.SIH\OceanScan")
OUT = ROOT / "output" / "noise_filter" / "train_noise_filter"
DATA_YAML = OUT / "data.yaml"
SUBSETS = ["train", "valid"]

names_block = DATA_YAML.read_text().split("names:")[-1].strip().splitlines()
NAMES = {}
for _line in names_block:
    _k, _v = _line.split(":", 1)
    NAMES[int(_k)] = _v.strip().strip('"')
print("Classes:", NAMES)

In [ ]:
print(DATA_YAML.read_text())
for sub in SUBSETS:
    n_img = len(list((OUT / sub / "images").glob("*")))
    n_lbl = len(list((OUT / sub / "labels").glob("*.txt")))
    print(f"{sub:35s} images={n_img}  labels={n_lbl}")

## 2. Train YOLOv8n with early stopping

Run `shutil.rmtree` on `backend/runs/dataset2_yolov8n` first to retrain from scratch.

In [ ]:
RUNS = ROOT / "backend" / "runs" / "dataset2_yolov8n"
# Delete a previous run to retrain from scratch:
shutil.rmtree(RUNS, ignore_errors=True)

model = YOLO("yolov8n.pt")  # downloads pretrained weights on first run

results = model.train(
    data=str(DATA_YAML),
    epochs=20,         # fixed 20 epochs to cap training time
    patience=15,       # auto early-stop: stop if val mAP50 stalls for 15 epochs
    imgsz=640,         # higher input size = better detection accuracy (slower on CPU)
    batch=8,
    device="cpu",
    workers=0,
    project=str(ROOT / "backend" / "runs"),
    name="dataset2_yolov8n",
    exist_ok=True,
    verbose=True,
)

## 3. Training curves

In [ ]:
csv_path = RUNS / "results.csv"
if not csv_path.exists():
    print("No results.csv yet - run the training cell above first.")
else:
    df = pd.read_csv(csv_path)
    cols = set(df.columns)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    if "metrics/mAP50(B)" in cols:
        axes[0].plot(df["epoch"], df["metrics/mAP50(B)"], label="val mAP50")
    if "metrics/mAP50-95(B)" in cols:
        axes[0].plot(df["epoch"], df["metrics/mAP50-95(B)"], label="val mAP50-95")
    axes[0].set(xlabel="epoch", title="Val metrics")
    if axes[0].lines:
        axes[0].legend()
    axes[0].grid(alpha=.3)
    if "train/box_loss" in cols:
        axes[1].plot(df["epoch"], df["train/box_loss"], label="train box")
    if "val/box_loss" in cols:
        axes[1].plot(df["epoch"], df["val/box_loss"], label="val box")
    axes[1].set(xlabel="epoch", title="Box loss")
    if axes[1].lines:
        axes[1].legend()
    axes[1].grid(alpha=.3)
    plt.tight_layout(); plt.show()

## 4. Evaluate best weights on the validation split

In [ ]:
best = RUNS / "weights" / "best.pt"
print("best.pt exists:", best.exists())
if best.exists():
    m = YOLO(str(best)).val(split="val", verbose=False)
    print(f"mAP50     : {m.box.map50:.4f}")
    print(f"mAP50-95  : {m.box.map:.4f}")
    for c in sorted(NAMES):
        print(f"  {NAMES[c]:6s} mAP50 = {m.box.maps[c]:.4f}")

## 5. Save model for the prediction notebook

In [ ]:
if best.exists():
    target = ROOT / "backend" / "best.pt"
    shutil.copy2(str(best), str(target))
    print(f"Saved -> {target}")
else:
    print("Training did not complete - no best.pt to save.")